# Compounding Intelligence — Colab v0

Falsification-first notebook for persistent learning, trajectory value, and governed admission.


In [ ]:
!pip -q install "transformers>=4.45" accelerate bitsandbytes pandas matplotlib


In [ ]:
from pathlib import Path
import json, random, re, time
from dataclasses import dataclass, asdict
from typing import List, Dict, Optional
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

SEED=7
random.seed(SEED); torch.manual_seed(SEED)
MODEL_ID="Qwen/Qwen2.5-1.5B-Instruct"
DTYPE=torch.float16 if torch.cuda.is_available() else torch.float32
RESULTS=Path("/content/results"); RESULTS.mkdir(parents=True, exist_ok=True)
print("device", "cuda" if torch.cuda.is_available() else "cpu")
print("model", MODEL_ID)


In [ ]:
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model=AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)
model.eval()

@torch.inference_mode()
def generate(prompt, max_new_tokens=96):
    messages=[
        {"role":"system","content":"Answer precisely. When asked for an integer, end with ANSWER=<integer>."},
        {"role":"user","content":prompt},
    ]
    text=tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs=tokenizer(text, return_tensors="pt")
    if torch.cuda.is_available():
        inputs={k:v.to(model.device) for k,v in inputs.items()}
    out=model.generate(**inputs,max_new_tokens=max_new_tokens,do_sample=False,pad_token_id=tokenizer.eos_token_id)
    new=out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new,skip_special_tokens=True)

def parse_int(text):
    m=re.search(r"ANSWER\\s*=\\s*(-?\\d+)", text)
    if m: return int(m.group(1))
    ints=re.findall(r"-?\\d+", text)
    return int(ints[-1]) if ints else None


## Synthetic hidden-rule world

The model is frozen. Holdout answers never enter memory.


In [ ]:
@dataclass(frozen=True)
class World:
    name:str
    rule_name:str
    def f(self,x):
        a,b,c=x
        if self.rule_name=="affine": return 2*a+3*b-c+5
        if self.rule_name=="interaction": return a*b+2*c-a
        if self.rule_name=="piecewise": return (a+b+c) if a>=b else (2*b-a+c)
        raise ValueError(self.rule_name)

WORLDS=[World("W1","affine"),World("W2","interaction"),World("W3","piecewise")]

def make_items(world,n_train=18,n_holdout=12,seed=SEED):
    r=random.Random((hash(world.name)^seed)&0xffffffff)
    seen=set(); items=[]
    while len(items)<n_train+n_holdout:
        x=(r.randint(-5,7),r.randint(-5,7),r.randint(-5,7))
        if x in seen: continue
        seen.add(x); items.append({"x":x,"y":world.f(x)})
    return items[:n_train],items[n_train:]
DATA={w.name:make_items(w) for w in WORLDS}


In [ ]:
@dataclass
class Episode:
    world:str
    episode_id:int
    x:tuple
    predicted:Optional[int]
    correct:int
    raw_response:str
    success:bool
    source_view:str

class GovernedState:
    def __init__(self):
        self._state={}
        self.events=[]
        self.direct_write_violations=0
    def read(self,world): return self._state.get(world)
    def _commit_admitted(self,world,candidate,evidence):
        self._state[world]=candidate
        self.events.append({"type":"ADMIT","world":world,"candidate":candidate,"evidence":evidence,"ts":time.time()})
    def reject(self,world,candidate,evidence):
        self.events.append({"type":"REJECT","world":world,"candidate":candidate,"evidence":evidence,"ts":time.time()})
    def unresolved(self,world,candidate,evidence):
        self.events.append({"type":"UNRESOLVED","world":world,"candidate":candidate,"evidence":evidence,"ts":time.time()})
    def forbidden_direct_write(self,world,value):
        self.direct_write_violations+=1
        raise RuntimeError("NO_DIRECT_GOVERNED_LEARNING_WRITE_PATH violated")


In [ ]:
def render_memory(episodes,view="full",limit=8):
    eps=episodes[-limit:]
    if not eps: return "(none)"
    lines=[]
    for e in eps:
        if view=="output":
            lines.append(f"x={e.x} -> correct={e.correct}")
        else:
            lines.append(f"x={e.x}; attempted={e.predicted}; correct={e.correct}; outcome={'success' if e.success else 'failure_then_feedback'}")
    return "\\n".join(lines)

def ask(world_name,x,memory="",admitted_rule=None):
    context=[]
    if admitted_rule: context.append("ADMITTED LEARNING:\\n"+admitted_rule)
    if memory and memory!="(none)": context.append("PAST EXPERIENCE:\\n"+memory)
    ctx="\\n\\n".join(context) if context else "No prior experience is available."
    prompt=f"""You are solving a hidden deterministic mapping in world {world_name}.
Inputs are integer triples x=(a,b,c). Infer the mapping only from supplied experience.

{ctx}

Now solve x={x}.
Return a short explanation and end exactly with ANSWER=<integer>."""
    raw=generate(prompt)
    return parse_int(raw),raw

def make_candidate(world_name,episodes,view="full"):
    material=render_memory(episodes,view=view,limit=12)
    prompt=f"""Infer a concise reusable rule for hidden world {world_name} from these observed episodes.

{material}

Return ONLY a compact candidate rule. Do not claim certainty if evidence is contradictory."""
    return generate(prompt,max_new_tokens=120).strip()


## Governed admission

Candidate learning is replay-tested before it can become authoritative state.


In [ ]:
def replay_score(world_name,candidate,replay_items):
    if not replay_items: return 0.0
    ok=0
    for item in replay_items:
        pred,_=ask(world_name,item["x"],admitted_rule=candidate)
        ok+=int(pred==item["y"])
    return ok/len(replay_items)

def admit_candidate(state,world_name,candidate,replay_items,min_items=4,min_score=0.75):
    if len(replay_items)<min_items:
        evidence={"n":len(replay_items),"reason":"insufficient replay evidence"}
        state.unresolved(world_name,candidate,evidence); return "UNRESOLVED"
    sample=replay_items[-min(8,len(replay_items)):]
    score=replay_score(world_name,candidate,sample)
    evidence={"n":len(sample),"replay_score":score,"threshold":min_score}
    if score>=min_score:
        state._commit_admitted(world_name,candidate,evidence); return "ADMIT"
    state.reject(world_name,candidate,evidence); return "REJECT"


In [ ]:
def run_condition(world,condition,source_view="full",consolidate_every=6):
    train,holdout=DATA[world.name]
    history=[]; state=GovernedState(); rows=[]
    for i,item in enumerate(train):
        if condition=="STATELESS":
            memory=""; rule=None
        elif condition=="PERSISTENT_RAW":
            memory=render_memory(history,view=source_view); rule=None
        elif condition=="PERSISTENT_CONSOLIDATED":
            memory=render_memory(history,view=source_view,limit=4); rule=state.read(world.name)
        else: raise ValueError(condition)
        pred,raw=ask(world.name,item["x"],memory=memory,admitted_rule=rule)
        ep=Episode(world.name,i,item["x"],pred,item["y"],raw,pred==item["y"],source_view)
        history.append(ep)
        row={"phase":"train","world":world.name,"condition":condition,"source_view":source_view,"episode":i,"correct":int(ep.success)}
        if condition=="PERSISTENT_CONSOLIDATED" and (i+1)%consolidate_every==0:
            candidate=make_candidate(world.name,history,view=source_view)
            replay=[{"x":e.x,"y":e.correct} for e in history]
            row["admission_decision"]=admit_candidate(state,world.name,candidate,replay)
        rows.append(row)
    for j,item in enumerate(holdout):
        if condition=="STATELESS":
            memory=""; rule=None
        elif condition=="PERSISTENT_RAW":
            memory=render_memory(history,view=source_view); rule=None
        else:
            memory=render_memory(history,view=source_view,limit=4); rule=state.read(world.name)
        pred,_=ask(world.name,item["x"],memory=memory,admitted_rule=rule)
        rows.append({"phase":"holdout","world":world.name,"condition":condition,"source_view":source_view,"episode":j,"correct":int(pred==item["y"])})
    return rows,history,state


In [ ]:
RUN_FULL_MATRIX=False
worlds=WORLDS if RUN_FULL_MATRIX else WORLDS[:1]
configs=[
    ("STATELESS","full"),
    ("PERSISTENT_RAW","output"),
    ("PERSISTENT_RAW","full"),
    ("PERSISTENT_CONSOLIDATED","output"),
    ("PERSISTENT_CONSOLIDATED","full"),
]
all_rows=[]; all_trajectories=[]; states={}
for w in worlds:
    for condition,view in configs:
        print("RUN",w.name,condition,view)
        rows,hist,state=run_condition(w,condition,source_view=view)
        all_rows.extend(rows)
        all_trajectories.extend([asdict(e)|{"condition":condition} for e in hist])
        states[f"{w.name}:{condition}:{view}"]={
            "admitted_state":state._state,
            "events":state.events,
            "direct_write_violations":state.direct_write_violations,
        }
scores=pd.DataFrame(all_rows)
display(scores.groupby(["phase","world","condition","source_view"])["correct"].mean().reset_index())


In [ ]:
manifest={
    "experiment":"compounding-intelligence-colab-v0",
    "seed":SEED,
    "model_id":MODEL_ID,
    "model_frozen":True,
    "worlds":[asdict(w) for w in worlds],
    "conditions":configs,
    "holdout_answers_admitted":False,
    "invariant":"NO_DIRECT_GOVERNED_LEARNING_WRITE_PATH",
    "timestamp":time.time(),
}
(RESULTS/"manifest.json").write_text(json.dumps(manifest,indent=2))
with (RESULTS/"trajectories.jsonl").open("w") as f:
    for row in all_trajectories: f.write(json.dumps(row)+"\\n")
(RESULTS/"governed_state.json").write_text(json.dumps(states,indent=2))
scores.to_csv(RESULTS/"scores.csv",index=False)
summary=scores.groupby(["phase","condition","source_view"])["correct"].mean().reset_index()
(RESULTS/"summary.json").write_text(summary.to_json(orient="records",indent=2))
assert all(v["direct_write_violations"]==0 for v in states.values())
print("evidence package",RESULTS)
display(summary)


In [ ]:
import matplotlib.pyplot as plt
hold=scores[scores.phase=="holdout"].groupby(["condition","source_view"])["correct"].mean().reset_index()
labels=[f"{r.condition}\\n{r.source_view}" for r in hold.itertuples()]
plt.figure(figsize=(10,4))
plt.bar(labels,hold["correct"])
plt.ylim(0,1)
plt.ylabel("Blind holdout accuracy")
plt.title("Colab v0 — persistent learning signal")
plt.xticks(rotation=20,ha="right")
plt.show()


## Interpretation

Scale only if the persistent conditions beat stateless control on blind holdout, results repeat across seeds/worlds, trajectory premium survives, and direct governed-write violations remain exactly zero.

A positive result is a signal, not proof of a universal `Cmin` or recursive self-improvement.
